# 02 — Dataset Builder

Converts raw collected text into three training-ready datasets:

1. **SFT corpus** — `data/processed/sft_rf_domain.jsonl`  
   Chat-format pairs: `{instruction, input, output}` covering RF design Q&A,  
   circuit sizing, layout code generation, antenna design, link budgets.

2. **Continued pre-training corpus** — `data/processed/pretrain_rf.jsonl`  
   Raw RF text (books, papers, code) chunked for next-token prediction.

3. **PINN dataset** — `data/processed/pinn_antenna.csv`  
   Geometry parameters → S11/gain/efficiency for training the EM surrogate model.

In [1]:
!pip install anthropic tqdm pandas scikit-learn

Defaulting to user installation because normal site-packages is not writeable
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 2.2 MB/s  0:00:0036m-:--:--
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [anthropic]/4 [anthropic]

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: /usr/bin/python3 -m pip install --upgrade pip


In [2]:
import json, re, random, os
import pandas as pd
from pathlib import Path
from tqdm import tqdm

random.seed(42)

RAW_DIR = Path('../data/raw')
PROC_DIR = Path('../data/processed')
SYNTH_DIR = Path('../data/synthetic')
for d in [PROC_DIR, SYNTH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 1. Continued Pre-Training Corpus — Chunk Raw Text

In [ ]:
CHUNK_SIZE = 2048   # tokens ~ chars/4
CHUNK_CHARS = CHUNK_SIZE * 4
OVERLAP = 256 * 4

def chunk_text(text: str, chunk_size: int = CHUNK_CHARS, overlap: int = OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if len(chunk) > 200:  # skip tiny fragments
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

pretrain_records = []

# Books + arXiv text
for txt_file in list((RAW_DIR / 'texts').glob('*.txt')) + list((RAW_DIR / 'arxiv' / 'texts').glob('*.txt')):
    text = txt_file.read_text(errors='ignore')
    source = txt_file.stem
    for chunk in chunk_text(text):
        pretrain_records.append({'text': chunk, 'source': source})

# arXiv abstracts
abstracts_file = RAW_DIR / 'arxiv' / 'all_abstracts.txt'
if abstracts_file.exists():
    text = abstracts_file.read_text(errors='ignore')
    for chunk in chunk_text(text, chunk_size=1024*4, overlap=128*4):
        pretrain_records.append({'text': chunk, 'source': 'arxiv_abstracts'})

# RF code corpus
code_file = RAW_DIR / 'code_texts' / 'rf_code_corpus.py'
if code_file.exists():
    text = code_file.read_text(errors='ignore')
    for chunk in chunk_text(text):
        pretrain_records.append({'text': chunk, 'source': 'rf_code'})

random.shuffle(pretrain_records)
out = PROC_DIR / 'pretrain_rf.jsonl'
with open(out, 'w') as f:
    for r in pretrain_records:
        f.write(json.dumps(r) + '\n')

print(f'Pre-training corpus: {len(pretrain_records):,} chunks -> {out}')

## 2. SFT Dataset — Handcrafted RF Design Q&A Templates

These are expert-written instruction-response pairs covering the core RF design tasks.

In [ ]:
# === HANDCRAFTED RF EXPERT Q&A ===
# Each entry: {instruction, input (optional context), output}

HANDCRAFTED_QA = [
    # --- LNA Design ---
    {
        "instruction": "Design a Ka-band (28 GHz) cascode LNA in 65nm CMOS targeting NF < 3 dB and gain > 20 dB. Specify topology, transistor sizing, matching network approach, and bias conditions.",
        "input": "",
        "output": """Ka-band cascode LNA design in 65nm CMOS (28 GHz):

**Topology:** Inductively-degenerated common-source (CS) input stage + common-gate (CG) cascode to maximize gain-bandwidth and improve reverse isolation.

**Transistor sizing:**
- M1 (CS): W = 40 µm, L = 60 nm, Nf = 20 fingers (W/f = 2 µm)
  - At 28 GHz, optimal Wf ≈ 1/(2π·f·Rg·Cgs) → 2 µm fingers minimize Rg contribution to NF
- M2 (CG): W = 40 µm, L = 60 nm, same sizing to maintain current drive

**Bias:** Vdd = 1.2 V, IDS ≈ 10 mA (IDS/Wg ≈ 0.25 mA/µm), Vgs − Vth ≈ 0.15 V (near peak-fT region)

**Matching network:**
- Input: Ls (source degeneration, 80 pH) sets real part of Zin = 50 Ω; Lg (gate inductor, 0.3 nH) resonates out imaginary part at 28 GHz
- Inter-stage: transmission-line transformer or shunt inductor for impedance transformation between CS and CG
- Output: shunt L + series C matching to 50 Ω load

**Expected performance:**
- NF: 2.5–3.0 dB (dominated by M1 drain current noise: NF_min ≈ 1 + 2·γ·ω/ωT·√(gm·Rn))
- Gain: 20–22 dB
- S11 < −10 dB
- IIP3 ≈ −5 dBm (limited by M1 nonlinearity)
- PDC ≈ 12 mW

**Layout considerations:**
- Use multi-finger transistors with gate contacted on both sides to halve Rg
- Route inductors in top-metal (thick Cu) with minimum substrate coupling
- Add substrate contacts (deep N-well isolation if available) around LNA core
- Use differential shielded inductors to reject common-mode substrate noise"""
    },
    # --- Phased Array ---
    {
        "instruction": "Explain the phase shifter architectures used in Ka-band CMOS phased arrays for satellite receivers, with tradeoffs.",
        "input": "",
        "output": """Ka-band CMOS phase shifter architectures for satellite phased arrays:

**1. Switched delay lines (true-time delay, TTD)**
- Operation: Select between transmission line segments of different lengths via RF switches (CMOS or FET)
- Advantage: Frequency-independent phase shift (no squint) — critical for wideband satellite signals
- Disadvantage: Area scales with bandwidth and frequency; insertion loss ~3–8 dB at Ka-band

**2. LC all-pass networks (vector modulator)**
- Operation: 90° hybrid splits signal into I/Q; variable attenuators on each path; recombine
- Advantage: Continuous phase resolution, amplitude control simultaneously
- Disadvantage: NF degradation, complex calibration, ~6–10 dB insertion loss

**3. Reflective-type phase shifter (RTPS)**
- Operation: 90° hybrid + variable reflection loads (varactors or switched caps)
- Advantage: Low IL (~2–4 dB), compact, good linearity
- Disadvantage: Limited phase range (~180°), amplitude-phase coupling

**4. Magnetic-tuning phase shifter (used in ISSCC 2022 Ka-band chip)**
- Operation: Current-controlled ferromagnetic material (permalloy) integrated on-chip shifts inductor permeability → shifts resonance frequency → shifts phase
- Advantage: No active devices in signal path → very low NF impact; radiation-hard (no carrier trapping)
- Disadvantage: Requires special process integration; limited tuning range

**5. Active phase rotator (digital)**
- Operation: Digitally weight I/Q components using current-mode DACs before combination
- Advantage: Digital control, easy calibration, fine resolution (6–8 bit)
- Disadvantage: Highest power, NF ≈ LNA NF + 3–6 dB degradation

**Recommendation for LEO satellite receiver (Ka-band, 26–30 GHz):**
RTPS or switched capacitor networks for best NF/area/power tradeoff at 65nm node.
Magnetic-tuning if radiation hardness is the primary constraint."""
    },
    # --- Antenna ---
    {
        "instruction": "Design a microstrip patch antenna for Ka-band at 28 GHz on Rogers RO4003C substrate (εr=3.55, h=0.508mm). Give all dimensions.",
        "input": "",
        "output": """Microstrip patch antenna design for 28 GHz (Ka-band) on RO4003C:

**Substrate:** εr = 3.55, tanδ = 0.0027, h = 0.508 mm

**Step 1 — Patch width W:**
W = (c / 2f) × √(2 / (εr + 1))
W = (3×10⁸ / 2×28×10⁹) × √(2 / 4.55) = 5.357 mm × 0.663 = **3.55 mm**

**Step 2 — Effective permittivity εeff:**
εeff = (εr+1)/2 + (εr−1)/2 × (1 + 12h/W)^(-0.5)
εeff = 2.275 + 1.275 × (1 + 12×0.508/3.55)^(-0.5) = 2.275 + 1.275×0.611 = **3.054**

**Step 3 — Effective patch length Leff:**
Leff = c / (2f × √εeff) = 3×10⁸ / (56×10⁹ × 1.748) = **3.064 mm**

**Step 4 — Length extension ΔL:**
ΔL = 0.412h × (εeff+0.3)(W/h+0.264) / ((εeff−0.258)(W/h+0.8))
ΔL = 0.412×0.508 × (3.354×7.137) / (2.796×7.813) = 0.209 × 23.94/21.84 = **0.229 mm**

**Step 5 — Physical patch length L:**
L = Leff − 2ΔL = 3.064 − 0.458 = **2.61 mm**

**Final dimensions:** W = 3.55 mm, L = 2.61 mm

**Feed:** Inset microstrip feed
- Feed line width (50 Ω): w₀ = 1.16 mm (for RO4003C h=0.508mm)
- Inset depth y₀ = L/π × arccos(√(Rin/50)) where Rin ≈ 200 Ω → y₀ ≈ 0.87 mm
- Inset slot width: 0.2 mm

**Expected performance:**
- Resonant freq: 28 GHz ±2%
- Gain: ~8 dBi
- BW (−10 dB S11): ~800 MHz (2.9%)
- Radiation efficiency: ~85–90% (limited by tanδ loss)"""
    },
    # --- GLayout code generation ---
    {
        "instruction": "Write GLayout Python code to generate a simple RF bypass capacitor cell in GF180MCU PDK using gdsfactory.",
        "input": "",
        "output": """```python
import gdsfactory as gf
from glayout.flow.pdk.mappedpdk import MappedPDK
from glayout.flow.primitives.mimcap import mimcap


@gf.cell
def rf_bypass_cap(
    pdk: MappedPDK,
    width: float = 20.0,   # um
    height: float = 20.0,  # um
    fingers: int = 4,
) -> gf.Component:
    """RF bypass capacitor using MIM cap array.
    
    Target: ~1 pF at Ka-band for supply bypass / matching.
    MIM cap density in GF180: ~1 fF/µm², so 20×20 µm = 400 fF per finger.
    4 fingers in parallel → ~1.6 pF.
    """
    c = gf.Component()
    
    finger_width = width
    finger_height = height / fingers
    spacing = pdk.get_grule('capm')['min_separation'] if 'capm' in pdk.grules else 2.0
    
    ports_top = []
    ports_bot = []
    
    for i in range(fingers):
        cap = c << mimcap(pdk, width=finger_width, height=finger_height)
        cap.movey(i * (finger_height + spacing))
        ports_top.append(cap.ports['top_met_N'])
        ports_bot.append(cap.ports['bottom_met_S'])
    
    # Connect all top plates (RF signal) together
    c.add_port('RF', port=ports_top[0])
    # Connect all bottom plates (GND) together  
    c.add_port('GND', port=ports_bot[0])
    
    c.flatten()
    return c


if __name__ == '__main__':
    from glayout.flow.pdk.gf180_mapped import gf180
    cap = rf_bypass_cap(pdk=gf180, width=20, height=20, fingers=4)
    cap.show()  # Opens in KLayout
    print(f'Bypass cap area: {cap.area():.1f} µm²')
```"""
    },
    # --- Link budget ---
    {
        "instruction": "Calculate the link budget for a Ka-band (28 GHz) LEO satellite downlink at 550 km altitude. EIRP = 40 dBW, antenna gain = 35 dBi, data rate = 100 Mbps. Is the link closed?",
        "input": "",
        "output": """Ka-band LEO downlink link budget (28 GHz, 550 km altitude):

**Transmit side (satellite):**
- EIRP: 40 dBW

**Free-space path loss (FSPL):**
FSPL = 20·log₁₀(4πd/λ) = 20·log₁₀(4π × 550×10³ / (3×10⁸/28×10⁹))
λ = 10.7 mm → FSPL = 20·log₁₀(4π × 550000 / 0.0107) = 20·log₁₀(6.45×10⁸) = **176.2 dB**

**Atmospheric losses:**
- Rain fade (Ka-band, moderate rain zone, 99.9% availability): 3 dB
- Gaseous absorption (O₂ + H₂O at 28 GHz): 0.5 dB
- Total atmospheric: **3.5 dB**

**Receive side (ground terminal):**
- Rx antenna gain: 35 dBi (phased array, ~50 cm aperture)
- Rx system noise temperature: 290 K → Tsys = 290 K → N₀ = kTsys = −174 + 10·log(290/290) + NF
- Rx NF: 3 dB → Tsys_total ≈ 580 K → N₀ = −174 + 10·log(2) = **−171 dBW/Hz**
- G/T = 35 − 10·log(580) = 35 − 27.6 = **7.4 dB/K**

**Received C/N₀:**
C/N₀ = EIRP − FSPL − Latm + G/T − k
= 40 − 176.2 − 3.5 + 35 − 3 − (−228.6)
= **121.9 dBHz**

**Required C/N₀ for 100 Mbps:**
Using DVB-S2 with 8PSK, code rate 2/3 → Eb/N₀ required = 6.5 dB
C/N₀_required = Eb/N₀ + 10·log(Rb) = 6.5 + 80 = **86.5 dBHz**

**Margin:** 121.9 − 86.5 = **+35.4 dB** ✓ Link is CLOSED with large margin.

This margin allows for:
- More aggressive modulation (32APSK, 64APSK) to increase spectral efficiency
- Smaller/cheaper ground terminal antenna
- Reduced transmit power to extend satellite battery life"""
    },
    # --- Radiation hardening ---
    {
        "instruction": "What techniques are used to make CMOS RF ICs radiation-hardened for LEO satellite applications?",
        "input": "",
        "output": """Radiation hardening techniques for CMOS RF ICs in LEO (total dose ~10–100 krad/year):

**1. Layout-level hardening (RHBD — Radiation Hardened By Design)**
- **Enclosed gate transistors (ELT):** Transistor gate surrounds the drain completely, eliminating parasitic edge leakage paths caused by STI (shallow trench isolation) oxide charge buildup. Eliminates radiation-induced leakage current increase.
- **Guard rings:** P+ rings around NMOS and N+ rings around PMOS to collect minority carriers from heavy ion strikes, preventing latch-up.
- **Single-event upset (SEU) mitigation:** Triple modular redundancy (TMR) on digital control registers; interleaved flip-flop layout to spread critical nodes.

**2. Bias point selection**
- Radiation shifts Vth of NMOS negative (oxide charge) and PMOS positive. Design bias circuits to tolerate ±100 mV Vth shift without performance collapse.
- Use self-biased topologies with PTAT current references that track Vth shifts.

**3. Magnetic-tuning components**
- Avoid varactors (MOS capacitors are sensitive to oxide charge) in phase shifters — substitute with magnetically-tuned inductors (ISSCC 2022 technique).
- Use switched-capacitor arrays with metal-insulator-metal (MIM) caps instead of MOS caps.

**4. Process selection**
- Thin gate oxide processes (65 nm, 28 nm) are inherently more radiation tolerant (fewer oxide traps) than older thick-oxide processes.
- SOI processes offer improved latch-up immunity but introduce floating body effects.

**5. Shielding**
- On-chip: Deep N-well isolation under sensitive analog blocks
- Module-level: Tantalum or aluminum shielding attenuates proton/electron flux

**Key spec:** Target TID tolerance > 100 krad(Si), SEL immunity > 60 MeV·cm²/mg LET threshold."""
    },
]

print(f'Handcrafted Q&A entries: {len(HANDCRAFTED_QA)}')
for qa in HANDCRAFTED_QA:
    print(f'  - {qa["instruction"][:70]}...')

## 3. Convert AICircuit Dataset to Q&A Format

In [ ]:
AICIRCUIT_DIR = RAW_DIR / 'aicircuit'
aicircuit_qa = []

CIRCUIT_DESCRIPTIONS = {
    'lna': ('Low Noise Amplifier (LNA)', ['gain_db', 'nf_db', 'iip3_dbm', 'p1db_dbm', 'bw_ghz']),
    'pa':  ('Power Amplifier (PA)',       ['pout_dbm', 'pae_pct', 'gain_db', 'p1db_dbm']),
    'vco': ('Voltage Controlled Oscillator (VCO)', ['freq_ghz', 'phase_noise_dbc', 'kvco_mhz_v', 'pdiss_mw']),
    'mixer': ('Mixer',                   ['conversion_gain_db', 'nf_db', 'iip3_dbm', 'lo_leakage_dbm']),
    'rx':  ('Receiver chain',            ['gain_db', 'nf_db', 'iip3_dbm', 'bw_ghz']),
    'tx':  ('Transmitter chain',         ['pout_dbm', 'evm_pct', 'bw_ghz']),
}

if AICIRCUIT_DIR.exists():
    for csv_file in AICIRCUIT_DIR.rglob('*.csv'):
        circuit_key = csv_file.stem.lower()
        circuit_name, perf_cols = next(
            ((v) for k, v in CIRCUIT_DESCRIPTIONS.items() if k in circuit_key),
            ('RF Circuit', [])
        )
        try:
            df = pd.read_csv(csv_file)
            param_cols = [c for c in df.columns if c not in perf_cols]
            actual_perf = [c for c in perf_cols if c in df.columns]
            actual_param = [c for c in param_cols if c in df.columns]
            
            if not actual_param or not actual_perf:
                # Auto-detect: last N columns are performance metrics
                actual_param = list(df.columns[:len(df.columns)//2])
                actual_perf = list(df.columns[len(df.columns)//2:])
            
            for _, row in df.sample(min(200, len(df))).iterrows():
                param_str = ', '.join(f'{c}={row[c]:.4g}' for c in actual_param if c in row)
                perf_str = '\n'.join(f'- {c}: {row[c]:.4g}' for c in actual_perf if c in row)
                aicircuit_qa.append({
                    'instruction': f'Given these design parameters for a {circuit_name}, predict the circuit performance metrics.',
                    'input': f'Design parameters: {param_str}',
                    'output': f'Predicted performance:\n{perf_str}',
                    'source': f'aicircuit_{csv_file.stem}',
                })
        except Exception as e:
            print(f'  {csv_file.name}: {e}')

print(f'AICircuit Q&A pairs: {len(aicircuit_qa)}')

## 4. Convert HuggingFace EE Datasets

In [ ]:
hf_qa = []
HF_DIR = RAW_DIR / 'huggingface'

if HF_DIR.exists():
    for jsonl_file in HF_DIR.glob('*.jsonl'):
        with open(jsonl_file) as f:
            for line in f:
                try:
                    row = json.loads(line)
                    # STEM-AI EE format
                    if 'question' in row and 'answer' in row:
                        hf_qa.append({
                            'instruction': row['question'],
                            'input': row.get('context', ''),
                            'output': row['answer'],
                            'source': jsonl_file.stem,
                        })
                    # EEE-Bench format
                    elif 'problem' in row and 'solution' in row:
                        hf_qa.append({
                            'instruction': row['problem'],
                            'input': '',
                            'output': row['solution'],
                            'source': jsonl_file.stem,
                        })
                except Exception:
                    pass

print(f'HuggingFace Q&A pairs: {len(hf_qa)}')

## 5. Synthetic RF Q&A Generation via Claude API

Use Claude to generate additional RF design Q&A pairs from the book text chunks.

In [ ]:
import anthropic

client = anthropic.Anthropic()  # uses ANTHROPIC_API_KEY env var

SYSTEM_PROMPT = """You are an expert RF IC and antenna engineer with 20 years of experience 
designing Ka-band CMOS chips for satellite communications. Generate educational Q&A pairs 
that would help train an AI assistant to answer RF design questions accurately."""

RF_TOPICS = [
    "impedance matching networks at Ka-band using L-match, π-match, and transformer-based topologies",
    "noise figure analysis in cascaded RF systems using Friis formula",
    "CMOS LNA topologies: CS, CG, folded-cascode, noise-canceling — tradeoffs at mmWave",
    "phased array antenna beamforming: analog, digital, hybrid — for LEO satellite",
    "S-parameter measurement and interpretation: S11, S21, S12, S22 for RF circuits",
    "GDSFactory/GLayout Python code for RF passive components: inductors, MIM caps, transmission lines",
    "patch antenna design equations: width, length, feed, efficiency, bandwidth",
    "Ka-band satellite link budget: EIRP, FSPL, G/T, Eb/N0, rain fade margin",
    "VCO phase noise: Leeson's model, flicker noise upconversion, tank Q vs phase noise",
    "power amplifier linearity: IIP3, P1dB, ACPR, Doherty architecture for satellite TX",
    "radiation hardening of CMOS RF ICs: ELT transistors, guard rings, TID effects",
    "microstrip vs CPW vs stripline for Ka-band routing on PCB and chip",
    "CubeSat antenna constraints: size, mass, deployable arrays, Ka-band gain requirements",
    "PINN (Physics-Informed Neural Network) for antenna S-parameter surrogate modeling",
    "silicon inductor design: Q factor, self-resonance frequency, substrate shielding techniques",
]

def generate_rf_qa(topic: str, n_pairs: int = 5) -> list:
    prompt = f"""Generate {n_pairs} high-quality Q&A pairs about: {topic}

Format each pair as JSON:
{{"question": "...", "answer": "..."}}

Requirements:
- Questions should be specific and technical (not vague)
- Answers should include equations, numerical examples, design guidelines
- Focus on Ka-band (26-40 GHz) satellite IC context where applicable
- Include GLayout/gdsfactory code examples for layout-related topics

Output only the JSON objects, one per line."""
    
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=4096,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": prompt}],
    )
    
    pairs = []
    for line in response.content[0].text.strip().split('\n'):
        line = line.strip()
        if line.startswith('{'):
            try:
                obj = json.loads(line)
                pairs.append({
                    'instruction': obj.get('question', ''),
                    'input': '',
                    'output': obj.get('answer', ''),
                    'source': 'synthetic_claude',
                    'topic': topic,
                })
            except json.JSONDecodeError:
                pass
    return pairs

# Generate synthetic Q&A (costs API tokens — comment out to skip)
synthetic_qa = []
for topic in tqdm(RF_TOPICS, desc='Generating synthetic RF Q&A'):
    pairs = generate_rf_qa(topic, n_pairs=8)
    synthetic_qa.extend(pairs)
    print(f'  {topic[:50]}: {len(pairs)} pairs')

synth_path = SYNTH_DIR / 'synthetic_rf_qa.jsonl'
with open(synth_path, 'w') as f:
    for qa in synthetic_qa:
        f.write(json.dumps(qa) + '\n')
print(f'\nSynthetic Q&A: {len(synthetic_qa)} pairs -> {synth_path}')

## 6. Merge All Sources into Final SFT Dataset

In [ ]:
def to_chatml(qa: dict) -> dict:
    """Convert to ChatML / Qwen chat format."""
    user_content = qa['instruction']
    if qa.get('input', '').strip():
        user_content += f"\n\nContext:\n{qa['input']}"
    return {
        'messages': [
            {'role': 'system', 'content': 'You are an expert RF IC and antenna engineer specializing in Ka-band CMOS chip design for satellite communications. Provide detailed, accurate technical answers with equations and design guidelines.'},
            {'role': 'user', 'content': user_content},
            {'role': 'assistant', 'content': qa['output']},
        ],
        'source': qa.get('source', 'unknown'),
    }

all_qa = []
all_qa.extend([{**qa, 'source': 'handcrafted'} for qa in HANDCRAFTED_QA])
all_qa.extend(aicircuit_qa)
all_qa.extend(hf_qa)

# Load synthetic if exists
synth_path = SYNTH_DIR / 'synthetic_rf_qa.jsonl'
if synth_path.exists():
    with open(synth_path) as f:
        all_qa.extend(json.loads(line) for line in f if line.strip())

# Filter: require non-empty instruction + output, min length
all_qa = [
    qa for qa in all_qa
    if len(qa.get('instruction', '')) > 20
    and len(qa.get('output', '')) > 50
]

random.shuffle(all_qa)
chatml_records = [to_chatml(qa) for qa in all_qa]

# Train/val split: 95/5
split = int(0.95 * len(chatml_records))
train_records = chatml_records[:split]
val_records = chatml_records[split:]

for fname, records in [('sft_train.jsonl', train_records), ('sft_val.jsonl', val_records)]:
    path = PROC_DIR / fname
    with open(path, 'w') as f:
        for r in records:
            f.write(json.dumps(r) + '\n')
    print(f'{fname}: {len(records):,} records')

print(f'\nTotal SFT dataset: {len(chatml_records):,} examples')
print('Source breakdown:')
from collections import Counter
for src, cnt in Counter(r['source'] for r in chatml_records).most_common():
    print(f'  {src:30s}: {cnt:5d}')

## 7. PINN Dataset — Antenna Geometry → EM Performance

In [ ]:
import numpy as np

# Load Mendeley antenna dataset if available
ANTENNA_DIR = RAW_DIR / 'antenna_dataset'
pinn_df = None

for csv_file in ANTENNA_DIR.glob('*.csv'):
    pinn_df = pd.read_csv(csv_file)
    print(f'Loaded Mendeley antenna dataset: {pinn_df.shape}')
    print(pinn_df.describe())
    break

if pinn_df is None:
    print('Mendeley dataset not found — generating synthetic patch antenna data using analytical model')
    
    # Analytical patch antenna model (Pozar transmission line model)
    def patch_antenna_model(er, h_mm, W_mm, L_mm, f_ghz):
        c = 3e8
        f = f_ghz * 1e9
        h = h_mm * 1e-3
        W = W_mm * 1e-3
        L = L_mm * 1e-3
        
        # Effective permittivity
        eeff = (er+1)/2 + (er-1)/2 * (1 + 12*h/W)**-0.5
        
        # Resonant frequency
        DL = 0.412*h * (eeff+0.3)*(W/h+0.264) / ((eeff-0.258)*(W/h+0.8))
        f_res = c / (2*(L + 2*DL)*np.sqrt(eeff))
        
        # S11 at frequency f (simplified Lorentzian resonance)
        BW_frac = 3.77 * (er-1)/er**2 * h/L * W/L  # Pozar BW formula
        BW = f_res * BW_frac
        Q = f_res / BW
        s11_linear = abs((1j*(f-f_res)/f_res * 2*Q) / (1 + 1j*(f-f_res)/f_res * 2*Q))
        s11_db = 20*np.log10(max(s11_linear, 1e-6))
        
        # Gain estimate (includes substrate loss)
        rad_eff = 1 / (1 + 2*np.pi*f*h*np.sqrt(er)*0.0027/c)  # tanδ=0.0027
        gain_dbi = 10*np.log10(4*np.pi * W*L / (c/f)**2 * rad_eff)
        
        return f_res/1e9, s11_db, gain_dbi, rad_eff*100
    
    N = 10000
    rows = []
    for _ in range(N):
        er = np.random.uniform(2.2, 10.2)
        h  = np.random.uniform(0.2, 2.0)   # mm
        f  = np.random.uniform(5, 40)       # GHz
        lam_guided = (3e8 / (f*1e9)) / np.sqrt(er) * 1e3  # mm
        W  = np.random.uniform(0.6*lam_guided/2, 1.4*lam_guided/2)
        L  = np.random.uniform(0.4*lam_guided/2, 0.6*lam_guided/2)
        
        f_res, s11, gain, eff = patch_antenna_model(er, h, W, L, f)
        rows.append({'er': er, 'h_mm': h, 'W_mm': W, 'L_mm': L, 'f_ghz': f,
                     'f_res_ghz': f_res, 's11_db': s11, 'gain_dbi': gain, 'efficiency_pct': eff})
    
    pinn_df = pd.DataFrame(rows)
    print(f'Generated synthetic antenna dataset: {pinn_df.shape}')

pinn_df.to_csv(PROC_DIR / 'pinn_antenna.csv', index=False)
print(f'PINN dataset saved: {PROC_DIR / "pinn_antenna.csv"}')
pinn_df.describe()

## 8. Chipster + AnalogCoder → SFT Training Pairs

Load the pre-extracted PySpice SFT pairs from `01_pdf_extract.ipynb` Section 7  
and merge them into the main SFT corpus in ChatML format.

Sources:
- **Chipster** — your own PySpice analog generator code and notebooks  
- **AnalogCoder** — 24 analog circuit problems with expert PySpice solutions (AAAI 2025)


In [ ]:
# ── Load analog PySpice SFT pairs produced by 01_pdf_extract.ipynb §7 ───────
ANALOG_DIR = RAW_DIR / 'analog_pyspice'
analog_sft_path = ANALOG_DIR / 'sft_pairs.jsonl'

analog_qa = []
if analog_sft_path.exists():
    with open(analog_sft_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
                analog_qa.append(row)
            except Exception:
                pass
    print(f'Loaded {len(analog_qa)} analog PySpice SFT pairs from {analog_sft_path.name}')
    # Source breakdown
    from collections import Counter
    counts = Counter(r.get('source','?') for r in analog_qa)
    for src, n in counts.items():
        print(f'  {src}: {n} pairs')
else:
    print(f'Not found: {analog_sft_path}')
    print('Run 01_pdf_extract.ipynb first (Section 7 clones + extracts both repos).')
    analog_qa = []

# ── Convert to ChatML format (same as all_qa list used in Section 6) ─────────
def analog_to_chatml(row: dict) -> dict:
    instruction = row.get('instruction', '')
    code_output  = row.get('output', '')
    circuit_name = row.get('circuit', 'analog circuit')
    source       = row.get('source', 'unknown')

    system_msg = (
        "You are an expert analog circuit designer specializing in PySpice simulation. "
        "When given a circuit specification, write complete, runnable Python code using "
        "PySpice that defines the netlist, configures ngspice simulation, and extracts "
        "key performance metrics (gain, bandwidth, noise figure, power consumption, etc.)."
    )

    return {
        "messages": [
            {"role": "system",    "content": system_msg},
            {"role": "user",      "content": instruction},
            {"role": "assistant", "content": code_output},
        ],
        "source":  source,
        "circuit": circuit_name,
    }

analog_chatml = [analog_to_chatml(r) for r in analog_qa if r.get('output','').strip()]
print(f'\nConverted {len(analog_chatml)} pairs to ChatML format')

# ── Append to main SFT JSONL ─────────────────────────────────────────────────
SFT_PATH = PROC_DIR / 'sft_rf_domain.jsonl'
existing = 0
if SFT_PATH.exists():
    with open(SFT_PATH) as f:
        existing = sum(1 for _ in f)

with open(SFT_PATH, 'a') as f:   # append — don't overwrite existing SFT data
    for item in analog_chatml:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

total = existing + len(analog_chatml)
print(f'SFT file: {existing} existing + {len(analog_chatml)} analog pairs = {total} total')
print(f'  -> {SFT_PATH}')

# ── Also add PySpice corpus to pre-training JSONL ────────────────────────────
PRETRAIN_PATH = PROC_DIR / 'pretrain_rf.jsonl'
corpus_path = ANALOG_DIR / 'corpus.py'
pretrain_added = 0

if corpus_path.exists():
    CHUNK_CHARS = 8192
    OVERLAP      = 512

    corpus_text = corpus_path.read_text(errors='ignore')
    start = 0
    chunks = []
    while start < len(corpus_text):
        chunk = corpus_text[start:start + CHUNK_CHARS].strip()
        if len(chunk) > 200:
            chunks.append(chunk)
        start += CHUNK_CHARS - OVERLAP

    with open(PRETRAIN_PATH, 'a') as f:
        for chunk in chunks:
            f.write(json.dumps({'text': chunk, 'source': 'analog_pyspice'}) + '\n')
    pretrain_added = len(chunks)
    print(f'\nPre-train corpus: +{pretrain_added} analog PySpice chunks -> {PRETRAIN_PATH.name}')

print('\nSample ChatML pair:')
if analog_chatml:
    ex = analog_chatml[0]
    print(f'  circuit: {ex["circuit"]} | source: {ex["source"]}')
    print(f'  user:    {ex["messages"][1]["content"][:120]}...')
    print(f'  assistant (first 200 chars): {ex["messages"][2]["content"][:200]}...')


In [ ]:
print('=== Final Dataset Summary ===')
for f in sorted(PROC_DIR.glob('*')):
    size = f.stat().st_size
    lines = sum(1 for _ in open(f, errors='ignore')) if f.suffix in ('.jsonl', '.csv') else 0
    print(f'  {f.name:35s}  {lines:6d} rows  {size/1e6:6.1f} MB')
print('\nNext: Run 03_qwen_sft_lora.ipynb to fine-tune Qwen')